In [2]:
import pandas as pd
import joblib

DATA_PROCESSED = "../data/processed"

X_train = pd.read_csv(f"{DATA_PROCESSED}/X_train.csv")
X_test = pd.read_csv(f"{DATA_PROCESSED}/X_test.csv")
y_train = pd.read_csv(f"{DATA_PROCESSED}/y_train.csv").squeeze()
y_test = pd.read_csv(f"{DATA_PROCESSED}/y_test.csv").squeeze()

print("Data loaded successfully")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Data loaded successfully
X_train: (1176, 55)
X_test : (294, 55)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name} trained")

C:\Users\Mitalika\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained
Random Forest trained
Gradient Boosting trained


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, pos_label="Yes", zero_division=0),
        "Recall": recall_score(y_test, y_pred, pos_label="Yes", zero_division=0),
        "F1": f1_score(y_test, y_pred, pos_label="Yes", zero_division=0),
        "ROC-AUC": roc_auc_score(
            (y_test == "Yes").astype(int),
            y_prob
        )
    })

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC", ascending=False
)

print(results_df.round(3))

                 Model  Accuracy  Precision  Recall     F1  ROC-AUC
2    Gradient Boosting     0.857      0.667   0.213  0.323    0.797
1        Random Forest     0.850      0.667   0.128  0.214    0.784
0  Logistic Regression     0.871      0.909   0.213  0.345    0.777


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

best_model = models["Gradient Boosting"]

y_pred = best_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=["No Attrition", "Attrition"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

No Attrition       0.87      0.98      0.92       247
   Attrition       0.67      0.21      0.32        47

    accuracy                           0.86       294
   macro avg       0.77      0.60      0.62       294
weighted avg       0.84      0.86      0.82       294


Confusion Matrix:
[[242   5]
 [ 37  10]]


In [7]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob = best_model.predict_proba(X_test)[:, 1]

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]

for threshold in thresholds:
    y_pred_threshold = np.where(y_prob >= threshold, "Yes", "No")

    print(
        f"Threshold {threshold:.2f} | "
        f"Precision: {precision_score(y_test, y_pred_threshold, pos_label='Yes'):.3f} | "
        f"Recall: {recall_score(y_test, y_pred_threshold, pos_label='Yes'):.3f} | "
        f"F1: {f1_score(y_test, y_pred_threshold, pos_label='Yes'):.3f}"
    )

Threshold 0.50 | Precision: 0.667 | Recall: 0.213 | F1: 0.323
Threshold 0.45 | Precision: 0.565 | Recall: 0.277 | F1: 0.371
Threshold 0.40 | Precision: 0.500 | Recall: 0.277 | F1: 0.356
Threshold 0.35 | Precision: 0.429 | Recall: 0.319 | F1: 0.366
Threshold 0.30 | Precision: 0.442 | Recall: 0.404 | F1: 0.422
Threshold 0.25 | Precision: 0.446 | Recall: 0.532 | F1: 0.485


In [8]:
BEST_MODEL = models["Gradient Boosting"]
BEST_THRESHOLD = 0.25

print("Selected Model: Gradient Boosting")
print("Selected Threshold:", BEST_THRESHOLD)

Selected Model: Gradient Boosting
Selected Threshold: 0.25


In [10]:
print("encoder" in globals())

False


In [11]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(BEST_MODEL, MODEL_DIR / "attrition_model.pkl")

print("Model saved successfully.")

Model saved successfully.


In [12]:
# Use the selected Gradient Boosting model
best_model = BEST_MODEL

# Predict probability of attrition
risk_probability = best_model.predict_proba(X_test)[:, 1]

# Create result table
risk_results = pd.DataFrame({
    "AttritionProbability": risk_probability,
    "ActualAttrition": y_test.values
}, index=X_test.index)

# Apply selected threshold
risk_results["PredictedAttrition"] = (
    risk_results["AttritionProbability"] >= BEST_THRESHOLD
).astype(int)

# Risk categories
risk_results["RiskLevel"] = pd.cut(
    risk_results["AttritionProbability"],
    bins=[-0.01, 0.25, 0.50, 1.0],
    labels=["Low", "Medium", "High"]
)

print(risk_results.head())
print("\nRisk distribution:")
print(risk_results["RiskLevel"].value_counts())

   AttritionProbability ActualAttrition  PredictedAttrition RiskLevel
0              0.287895              No                   1    Medium
1              0.022285              No                   0       Low
2              0.117092              No                   0       Low
3              0.009519              No                   0       Low
4              0.283012             Yes                   1    Medium

Risk distribution:
RiskLevel
Low       238
Medium     41
High       15
Name: count, dtype: int64


In [14]:
import pandas as pd

attrition_hr = pd.read_csv(
    "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"
)

risk_results["EmployeeNumber"] = attrition_hr.loc[
    risk_results.index, "EmployeeNumber"
].values

risk_results = risk_results[
    [
        "EmployeeNumber",
        "AttritionProbability",
        "PredictedAttrition",
        "RiskLevel",
        "ActualAttrition"
    ]
]

print(risk_results.head(10))

   EmployeeNumber  AttritionProbability  PredictedAttrition RiskLevel  \
0               1              0.287895                   1    Medium   
1               2              0.022285                   0       Low   
2               4              0.117092                   0       Low   
3               5              0.009519                   0       Low   
4               7              0.283012                   1    Medium   
5               8              0.092380                   0       Low   
6              10              0.071071                   0       Low   
7              11              0.050783                   0       Low   
8              12              0.015427                   0       Low   
9              13              0.434828                   1    Medium   

  ActualAttrition  
0              No  
1              No  
2              No  
3              No  
4             Yes  
5              No  
6              No  
7              No  
8              N

In [16]:
risk_results.to_csv(
    "../data/processed/attrition_risk_predictions.csv",
    index=False
)

print("Risk predictions saved successfully.")

Risk predictions saved successfully.


In [17]:
print(risk_results.shape)
print(risk_results["RiskLevel"].value_counts())

(294, 5)
RiskLevel
Low       238
Medium     41
High       15
Name: count, dtype: int64
